# Week 16: Integrating Agentic AI — Strands Agents & Amazon Bedrock AgentCore

## Learning Objectives

By the end of this session, you will be able to:
1. **Build agents with Strands Agents SDK** — AWS's production agent framework
2. **Create multi-agent systems** using the agents-as-tools pattern
3. **Use AgentCore Memory** for persistent, cross-session agent knowledge
4. **Understand the production stack** — how agents go from notebook to deployment

## Prerequisites

- Completed Week 15 (ReAct agents, LangChain `create_react_agent`, tools, memory)
- Completed Week 13 (Amazon Bedrock Converse API, boto3 setup)
- Watched pre-class videos on multi-agent architectures, orchestration patterns

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Recap | 10 min | Code |
| Section 1: Strands Agents — From LangChain to AWS-Native | 20 min | Demo |
| Lab 1: Build a Fraud Agent with Strands | 15 min | Lab |
| Section 2: Multi-Agent Orchestration | 25 min | Demo |
| Lab 2: Build a Multi-Agent Fraud Pipeline | 15 min | Lab |
| Section 3: AgentCore Memory & Production Services | 15 min | Demo |
| Wrap-up & Homework | 5 min | Markdown |

## The Story So Far

In Week 15, you built your first AI agent — a fraud investigator that could
reason, call tools, and solve multi-step problems. But you built it on your
laptop with LangChain. Now imagine you want to deploy that agent
to handle thousands of fraud cases per day. You need:

- **A production framework** that's AWS-native and battle-tested
- **Persistent memory** so the agent remembers past investigations
- **Multi-agent coordination** so specialized agents collaborate on complex cases
- **Managed infrastructure** so you don't babysit containers

That's exactly what **Strands Agents SDK** + **Amazon Bedrock AgentCore** provide.

## No GPU Needed

All work is API-based through Amazon Bedrock — no GPU required.

# Section 0: Environment Setup

We'll use **Strands Agents SDK** (AWS's open-source agent framework) and
the **Amazon Bedrock AgentCore SDK** for production memory services.

Both work seamlessly with Amazon Bedrock models — no additional API keys needed
beyond your AWS credentials from Week 13.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# strands-agents: AWS's open-source agent framework
# strands-agents-tools: Pre-built tools (calculator, web search, etc.)
# bedrock-agentcore: AgentCore SDK (Memory, Runtime, Gateway)
# Note: boto3 and sagemaker are pre-installed on SageMaker

!pip install -q strands-agents strands-agents-tools bedrock-agentcore

# =============================================================================
# IMPORTS
# =============================================================================
import os
import json
import boto3
import sagemaker
from sagemaker import get_execution_role
from importlib.metadata import version
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import Swarm              # Swarm pattern (Section 2 bonus)

# AgentCore Memory — low-level API
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

# AgentCore Memory — Strands integration (auto stores/retrieves turns)
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
print("Library versions:")
print(f"  strands-agents:    {version('strands-agents')}")
print(f"  bedrock-agentcore: {version('bedrock-agentcore')}")
print(f"  boto3:             {boto3.__version__}")
print(f"  sagemaker:         {sagemaker.__version__}")
print("\nAll libraries installed successfully!")

In [ ]:
# =============================================================================
# SAGEMAKER + BEDROCK CONNECTION SETUP
# =============================================================================
# On SageMaker, we get AWS credentials automatically through the execution role.
# No need for getpass or manual credential entry — the IAM role attached to this
# notebook instance already has permissions for Bedrock.

sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

print(f"SageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region: {AWS_REGION}")

# Verify Bedrock connection
bedrock_client = boto3.client('bedrock', region_name=AWS_REGION)
try:
    models = bedrock_client.list_foundation_models()
    model_count = len(models['modelSummaries'])
    print(f"\n✅ Connected to Bedrock! {model_count} models available.")
except Exception as e:
    print(f"\n❌ Bedrock connection failed: {e}")
    print("Ask your instructor to verify the execution role has Bedrock permissions.")

# =============================================================================
# MODEL CONFIGURATION
# =============================================================================
# Same Claude Haiku model from Week 15 — fast, cheap, great at tool use
MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"

# Create Strands BedrockModel using SageMaker's IAM role (automatic auth)
llm = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
)
print(f"\nAgent LLM: {MODEL_ID}")
print(f"Same Claude Haiku model from Week 15 — fast, cheap, supports tool use.")

In [ ]:
# =============================================================================
# FRAUD INVESTIGATION DATA — Same Database as Week 15
# =============================================================================
# We reuse the EXACT same fraud transactions from Week 15. The tools change
# (Strands instead of LangChain), but the data stays the same.

TRANSACTION_DATABASE = {
    "TXN-001": {"id": "TXN-001", "amount": 4500, "merchant": "Unknown Overseas Account", "type": "wire_transfer", "time": "03:47", "location": "International", "description": "Customer reports unauthorized wire transfer of $4,500 to unknown overseas account. No prior international transaction history. Transfer initiated at 3:47 AM local time."},
    "TXN-002": {"id": "TXN-002", "amount": 89.99, "merchant": "Netflix", "type": "subscription", "time": "10:00", "location": "Online", "description": "Regular monthly payment of $89.99 to Netflix streaming service. Consistent with 18-month subscription history. Payment from primary checking account."},
    "TXN-003": {"id": "TXN-003", "amount": 1500, "merchant": "Multiple ATMs", "type": "atm_withdrawal", "time": "14:30", "location": "Multiple cities", "description": "Three consecutive ATM withdrawals totaling $1,500 in different cities within 2 hours. Card was reported lost the following day. Withdrawals at non-bank ATMs."},
    "TXN-004": {"id": "TXN-004", "amount": 234.56, "merchant": "Amazon.com", "type": "online_purchase", "time": "15:20", "location": "Online", "description": "Online purchase of $234.56 at Amazon.com for household electronics. Shipping to address on file. Customer has frequent Amazon purchase history."},
    "TXN-005": {"id": "TXN-005", "amount": 2100, "merchant": "Luxury Jewelry Store", "type": "in_store", "time": "11:30", "location": "Miami", "description": "Customer disputes charge of $2,100 at luxury jewelry store in Miami. Customer's location confirmed as Chicago at time of purchase. No travel alerts set."},
    "TXN-006": {"id": "TXN-006", "amount": 3245.67, "merchant": "ABC Corp", "type": "direct_deposit", "time": "06:00", "location": "N/A", "description": "Automatic payroll direct deposit of $3,245.67 from employer ABC Corp. Matches bi-weekly pay schedule. Amount consistent with employment records."},
    "TXN-007": {"id": "TXN-007", "amount": 87.50, "merchant": "Various Digital Stores", "type": "online_purchase", "time": "22:15", "location": "Online", "description": "Multiple small online purchases ($5-$15) at various digital stores within 30 minutes. None of these merchants appear in customer's history. Different IP addresses used."},
    "TXN-008": {"id": "TXN-008", "amount": 67.23, "merchant": "Whole Foods Market", "type": "in_store", "time": "17:45", "location": "Home area", "description": "Grocery purchase of $67.23 at Whole Foods Market. Customer shops here weekly based on 2-year transaction history. Paid with debit card at POS terminal."},
    "TXN-009": {"id": "TXN-009", "amount": 8200, "merchant": "New Payee Transfer", "type": "wire_transfer", "time": "02:30", "location": "Foreign IP", "description": "Account password changed and $8,200 transferred to a new payee within 15 minutes. Login originated from an IP address in a different country than the account holder's residence."},
    "TXN-010": {"id": "TXN-010", "amount": 3400, "merchant": "Electronics Store Lagos", "type": "in_store", "time": "16:00", "location": "Lagos, Nigeria", "description": "Credit card used for $3,400 purchase at electronics store in Lagos, Nigeria. Cardholder has never traveled outside the United States. Card was not reported stolen."},
}

CUSTOMER_HISTORY = {
    "TXN-001": {"avg_monthly_spend": 2500, "international_transactions": 0, "account_age_years": 5, "typical_hours": "8:00-22:00", "flagged_before": False},
    "TXN-002": {"avg_monthly_spend": 3200, "international_transactions": 0, "account_age_years": 3, "typical_hours": "7:00-23:00", "flagged_before": False},
    "TXN-003": {"avg_monthly_spend": 1800, "international_transactions": 2, "account_age_years": 7, "typical_hours": "9:00-21:00", "flagged_before": True},
    "TXN-004": {"avg_monthly_spend": 4100, "international_transactions": 5, "account_age_years": 10, "typical_hours": "6:00-00:00", "flagged_before": False},
    "TXN-005": {"avg_monthly_spend": 3500, "international_transactions": 1, "account_age_years": 4, "typical_hours": "8:00-22:00", "flagged_before": False},
    "TXN-006": {"avg_monthly_spend": 5000, "international_transactions": 3, "account_age_years": 8, "typical_hours": "6:00-23:00", "flagged_before": False},
    "TXN-007": {"avg_monthly_spend": 1200, "international_transactions": 0, "account_age_years": 2, "typical_hours": "9:00-21:00", "flagged_before": False},
    "TXN-008": {"avg_monthly_spend": 2800, "international_transactions": 1, "account_age_years": 6, "typical_hours": "7:00-22:00", "flagged_before": False},
    "TXN-009": {"avg_monthly_spend": 2200, "international_transactions": 0, "account_age_years": 4, "typical_hours": "8:00-20:00", "flagged_before": False},
    "TXN-010": {"avg_monthly_spend": 1500, "international_transactions": 0, "account_age_years": 3, "typical_hours": "9:00-21:00", "flagged_before": False},
}

FRAUD_POLICIES = {
    "international_first_time": "Flag and hold any first-time international transaction over $500. Require customer verification within 24 hours.",
    "unusual_hours": "Transactions between 1:00 AM and 5:00 AM outside customer's typical pattern require enhanced monitoring.",
    "velocity_check": "More than 3 transactions within 30 minutes at different merchants triggers automatic review.",
    "geographic_mismatch": "Transaction location more than 500 miles from customer's last known location within 2 hours requires hold.",
    "amount_threshold": "Single transactions exceeding 3x the customer's average monthly spend require supervisor approval.",
    "new_payee_large_transfer": "Wire transfers over $5,000 to newly added payees require two-factor verification and 24-hour hold.",
    "card_testing_pattern": "Multiple small transactions ($0.01-$5.00) at different merchants within 10 minutes indicate card testing.",
}

print(f"Transaction database: {len(TRANSACTION_DATABASE)} transactions")
print(f"Customer histories:   {len(CUSTOMER_HISTORY)} records")
print(f"Fraud policies:       {len(FRAUD_POLICIES)} rules")
print(f"\nSample transaction IDs: {list(TRANSACTION_DATABASE.keys())[:5]}...")
print(f"\n💡 Same data as Week 15 — the tools change, the data stays the same.")

# Section 1: Strands Agents — From LangChain to AWS-Native

## Framework Comparison

In Week 15, we used **LangChain + LangGraph** to build agents:

```python
# Week 15 (LangChain)
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

llm = ChatBedrockConverse(model="us.anthropic.claude-3-haiku-20240307-v1:0")
agent = create_react_agent(llm, tools=[my_tool], checkpointer=MemorySaver())
result = agent.invoke({"messages": [("user", "Investigate TXN-001")]},
                      config={"configurable": {"thread_id": "session-1"}})
```

This week, we switch to **Strands Agents** — AWS's production framework:

```python
# Week 16 (Strands)
from strands import Agent, tool
from strands.models import BedrockModel

llm = BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0")
agent = Agent(model=llm, tools=[my_tool], system_prompt="You are a fraud investigator.")
response = agent("Investigate TXN-001")
```

**Notice**: Same concepts (model, tools, system prompt), cleaner API.

## Why Strands for Production?

| Feature | LangChain (Week 15) | Strands (Week 16) |
|---------|---------------------|-------------------|
| Maintained by | LangChain Inc. | AWS |
| Agent creation | `create_react_agent(llm, tools, ...)` | `Agent(model, tools, ...)` |
| Tool decorator | `@tool` from `langchain_core` | `@tool` from `strands` |
| Memory | `MemorySaver` (in-process) | AgentCore Memory (persistent, managed) |
| Multi-agent | LangGraph StateGraph | Agents-as-tools pattern |
| Deployment | Self-managed | AgentCore Runtime (managed) |
| AWS integration | Via `langchain-aws` adapter | Native |

**Key point**: The CONCEPTS are the same — ReAct loop, tools, memory. What
changes is the framework. This is normal in software: you learn the pattern
once, then apply it in whatever toolkit your company uses.

In [ ]:
# =============================================================================
# DEMO: Rebuild the Fraud Investigation Agent in Strands
# =============================================================================
# In Week 15, we built fraud tools with LangChain's @tool. Now the SAME tools
# in Strands — notice how similar the @tool decorator is.
# The tools read from the SAME global constants (TRANSACTION_DATABASE, etc.)

# --- Tool 1: Look up a transaction ---
@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID from the fraud database.

    Args:
        transaction_id: The transaction ID to look up (e.g., 'TXN-001')
    """
    txn = TRANSACTION_DATABASE.get(transaction_id)
    if txn:
        return json.dumps(txn, indent=2)
    return f"Transaction {transaction_id} not found in database."

# --- Tool 2: Check customer history ---
@tool
def check_customer_history(transaction_id: str) -> str:
    """Check customer spending history and patterns for a given transaction.

    Args:
        transaction_id: The transaction ID to check history for
    """
    history = CUSTOMER_HISTORY.get(transaction_id)
    if history:
        return json.dumps(history, indent=2)
    return f"No customer history found for {transaction_id}."

# --- Tool 3: Calculate risk score ---
@tool
def calculate_risk_score(amount: float, avg_monthly_spend: float,
                         is_international: bool, is_unusual_hour: bool,
                         is_new_merchant: bool) -> str:
    """Calculate a fraud risk score (0-100) based on transaction characteristics.

    Args:
        amount: Transaction amount in dollars
        avg_monthly_spend: Customer's average monthly spending
        is_international: Whether the transaction crosses borders
        is_unusual_hour: Whether the transaction occurred at an unusual time
        is_new_merchant: Whether the merchant is new to this customer
    """
    # Same weighted formula as Week 15
    amount_ratio = min(amount / max(avg_monthly_spend, 1), 5.0)
    amount_score = min(amount_ratio * 20, 100)

    intl_score = 90 if is_international else 0
    hour_score = 70 if is_unusual_hour else 0
    merchant_score = 50 if is_new_merchant else 0

    risk_score = (
        amount_score * 0.30 +
        intl_score * 0.25 +
        hour_score * 0.20 +
        merchant_score * 0.15 +
        10  # Base risk
    )

    risk_level = "LOW" if risk_score < 30 else "MEDIUM" if risk_score < 60 else "HIGH"

    return json.dumps({
        "risk_score": round(risk_score, 1),
        "risk_level": risk_level,
        "components": {
            "amount_ratio_score": round(amount_score * 0.30, 1),
            "international_score": round(intl_score * 0.25, 1),
            "unusual_hour_score": round(hour_score * 0.20, 1),
            "new_merchant_score": round(merchant_score * 0.15, 1),
            "base_risk": 10
        }
    }, indent=2)

# --- Tool 4: Check fraud policy ---
@tool
def check_fraud_policy(policy_type: str) -> str:
    """Look up a specific fraud prevention policy by type.

    Args:
        policy_type: The policy type to look up (e.g., 'international_first_time', 'unusual_hours', or 'all')
    """
    if policy_type == "all":
        return json.dumps(FRAUD_POLICIES, indent=2)

    policy = FRAUD_POLICIES.get(policy_type)
    if policy:
        return json.dumps({"policy_type": policy_type, "rule": policy}, indent=2)
    return f"Policy '{policy_type}' not found. Available: {list(FRAUD_POLICIES.keys())}"

# --- Create the agent ---
fraud_tools = [lookup_transaction, check_customer_history,
               calculate_risk_score, check_fraud_policy]

fraud_agent = Agent(
    model=llm,
    tools=fraud_tools,
    system_prompt=(
        "You are a senior fraud analyst at a major financial institution. "
        "When investigating a transaction, always:\n"
        "1) Look up the transaction details first\n"
        "2) Check the customer's spending history\n"
        "3) Calculate a risk score based on what you found\n"
        "4) Check relevant fraud policies\n"
        "5) Provide a clear VERDICT (FRAUD / LEGITIMATE / NEEDS REVIEW) "
        "with detailed reasoning.\n\n"
        "Be thorough but concise. Cite specific evidence from the tools."
    ),
)

print("✅ Fraud investigation agent created with Strands!")
print(f"   Model: {MODEL_ID}")
print(f"   Tools: {[t.tool_name for t in fraud_tools]}")
print(f"\n💡 Compare this to Week 15's LangChain agent — same tools, simpler setup.")

In [ ]:
# =============================================================================
# DEMO: Run the Fraud Agent on a Suspicious Transaction
# =============================================================================

print("Investigating TXN-001 (suspected fraud)...")
print("=" * 60)

response = fraud_agent("Investigate transaction TXN-001 for customer CUST-001. "
                       "Determine if this is fraud and what action to take.")

print("=" * 60)
print(f"\nAgent's response:\n{response}")

In [ ]:
# =============================================================================
# DEMO: Same Task, Different Framework — Same Result
# =============================================================================
# The point: agent concepts are PORTABLE. Tools, reasoning, memory — these are
# patterns, not framework-specific features.

print("Framework Comparison:")
print()
print("LangChain (Week 15)                    Strands (Week 16)")
print("-" * 70)
print("from langchain_core.tools import tool   from strands import tool")
print("from langchain_aws import               from strands.models import")
print("    ChatBedrockConverse                     BedrockModel")
print("from langgraph.prebuilt import          agent = Agent(")
print("    create_react_agent                      model=llm,")
print("agent = create_react_agent(                 tools=[...],")
print("    llm, tools=[...],                       system_prompt='...'")
print("    prompt='...'                         )")
print(")")
print()
print("Key differences:")
print("  * Strands: Agent() constructor — all config in one place")
print("  * Strands: Response is direct string, not message dict")
print("  * Strands: @tool decorator auto-parses docstring Args section")
print("  * Both: Same ReAct loop under the hood (think -> act -> observe)")
print("  * Both: Same Bedrock model, same tool-calling protocol")

> **Think About It**: You just rebuilt the SAME fraud agent in a different
> framework — LangChain in Week 15, Strands in Week 16. The tools are identical,
> the model is the same, the reasoning is the same. What does this tell you
> about investing too heavily in one framework? What criteria would you use
> to choose between frameworks for a production system?

## Lab 1: Build a Fraud Agent with Strands (15 minutes)

### Your Task

Build a Strands agent with a NEW tool and a customized system prompt for a
specific fraud investigation scenario.

### Steps

1. **Create a new tool** called `get_similar_transactions` that takes a
   `merchant_name` and returns a list of recent transactions at that merchant.
   Use the `@tool` decorator with full docstring (including Args section).
2. **Create an Agent** with ALL 5 tools (the 4 from the demo + your new one),
   a Bedrock model, and a system prompt that instructs the agent to always
   check for similar transactions before rendering a verdict.
3. **Test your agent** on a transaction at an unknown merchant — does it use
   your new tool?

### Expected Output

- A working `get_similar_transactions` tool
- An Agent that calls all relevant tools
- A verdict that references similar transaction patterns

### Hints

- The `@tool` decorator works exactly like LangChain's — decorate a function,
  add type hints, write a docstring with `Args:` section
- Return a JSON string from your tool (use `json.dumps()`)
- Test with: `agent("Investigate a $3,200 purchase at CryptoExchange for CUST-001")`

### Homework Extension

After class: add a 6th tool that wraps your Week 14 fine-tuned DistilBERT
model as a fraud classifier tool. Load the model with `AutoModelForSequenceClassification`
and expose it as a `@tool` that takes a transaction description and returns
the predicted label + confidence.

In [ ]:
# =============================================================================
# LAB 1: BUILD A FRAUD AGENT WITH STRANDS
# =============================================================================

# Step 1: Create the get_similar_transactions tool
@tool
def get_similar_transactions(merchant_name: str) -> str:
    """Find recent transactions at the same merchant across all customers.

    Args:
        merchant_name: The merchant name to search for
    """
    result = None  # YOUR CODE

    return json.dumps(result, indent=2)


# Step 2: Create an Agent with all 5 tools
all_tools = None  # YOUR CODE

lab1_agent = None  # YOUR CODE


# Step 3: Test the agent
test_response = None  # YOUR CODE

print(test_response)

In [ ]:
# =============================================================================
# FALLBACK: Ensures Section 2 works even if you didn't complete Lab 1
# =============================================================================
# The multi-agent demo below needs get_similar_transactions. If your Lab 1
# version isn't working yet, this provides a working implementation.
# If you DID complete Lab 1, this just redefines the same tool — no harm.

@tool
def get_similar_transactions(transaction_type: str) -> str:
    """Find all transactions of a given type to identify patterns.

    Args:
        transaction_type: The transaction type to search for (e.g., 'wire_transfer')
    """
    similar = [
        {"id": txn["id"], "amount": txn["amount"], "description": txn["description"][:100]}
        for txn in TRANSACTION_DATABASE.values()
        if txn["type"] == transaction_type
    ]
    if similar:
        return json.dumps(similar, indent=2)
    return f"No transactions of type '{transaction_type}' found."

print(f"get_similar_transactions tool ready ({len(TRANSACTION_DATABASE)} transactions searchable)")

# Section 2: Multi-Agent Orchestration

## The Agents-as-Tools Pattern

In production fraud operations, you don't have ONE agent doing everything.
You have **specialized agents** that each handle a piece of the pipeline:

```
+------------------------------------------------------------------+
|                    SUPERVISOR AGENT                                |
|  "Coordinate the fraud investigation — delegate to specialists"   |
|                                                                    |
|  +----------------+  +----------------+  +----------------+       |
|  | Triage Agent   |  | Investigation  |  | Decision       |       |
|  | (classify      |  | Agent          |  | Agent          |       |
|  |  risk level)   |  | (gather        |  | (render        |       |
|  |                |  |  evidence)     |  |  verdict)      |       |
|  +----------------+  +----------------+  +----------------+       |
+------------------------------------------------------------------+
```

**How it works in Strands**:
1. Each specialist is a regular `Agent()` with its own tools and system prompt
2. Wrap each specialist in a `@tool` decorator — now it's callable like a function
3. Give all specialist-tools to a supervisor `Agent()`
4. The supervisor decides which specialists to call and in what order

This is called the **agents-as-tools** pattern. It's simple, powerful, and
the recommended approach in Strands for multi-agent systems.

In [ ]:
# =============================================================================
# DEMO: Define Specialist Agents for the Fraud Pipeline
# =============================================================================
# Each specialist agent has its own tools, system prompt, and expertise.
# We'll then wrap them as tools for a supervisor agent.

# --- Specialist 1: Triage Agent ---
# Fast, rule-based classification. Uses calculate_risk_score tool.
triage_agent = Agent(
    model=llm,
    tools=[lookup_transaction, calculate_risk_score],
    system_prompt=(
        "You are a fraud triage specialist. Your ONLY job is to quickly "
        "classify transactions by risk level. "
        "Look up the transaction, calculate the risk score, and return "
        "a JSON-formatted triage result with: transaction_id, risk_level "
        "(LOW/MEDIUM/HIGH), risk_score, and key_factors. "
        "Be concise — no lengthy analysis. Speed matters in triage."
    ),
    # Suppress streaming output from sub-agents to keep supervisor output clean
    callback_handler=None,
)

# --- Specialist 2: Investigation Agent ---
# Deep dive into suspicious transactions. Uses all evidence-gathering tools.
investigation_agent = Agent(
    model=llm,
    tools=[lookup_transaction, check_customer_history,
           get_similar_transactions],
    system_prompt=(
        "You are a fraud investigation specialist. You receive transactions "
        "that have been flagged as MEDIUM or HIGH risk. Your job is to: "
        "1) Gather all available evidence (transaction details, customer history, "
        "   similar transactions at the same merchant) "
        "2) Write a detailed investigation report with findings "
        "3) Note any red flags or mitigating factors "
        "Be thorough — your report will be used for the final decision."
    ),
    callback_handler=None,
)

# --- Specialist 3: Decision Agent ---
# Renders final verdict based on triage + investigation reports.
decision_agent = Agent(
    model=llm,
    tools=[check_fraud_policy],
    system_prompt=(
        "You are a fraud decision specialist. You receive a triage result "
        "and an investigation report. Your job is to: "
        "1) Check the fraud policy for the given risk level and amount "
        "2) Render a FINAL VERDICT: APPROVE, FLAG_FOR_REVIEW, or BLOCK "
        "3) Provide a one-paragraph justification "
        "4) List the recommended actions "
        "Your decision is final and must be defensible in an audit."
    ),
    callback_handler=None,
)

print("✅ Three specialist agents created:")
print(f"   1. Triage Agent    — tools: lookup_transaction, calculate_risk_score")
print(f"   2. Investigation   — tools: lookup_transaction, check_customer_history, get_similar_transactions")
print(f"   3. Decision Agent  — tools: check_fraud_policy")
print(f"\n💡 callback_handler=None suppresses each agent's intermediate output.")
print(f"   Only the supervisor's output will be visible to the user.")

In [ ]:
# =============================================================================
# DEMO: Wrap Specialist Agents as Tools & Create Supervisor
# =============================================================================
# This is the KEY pattern: each specialist agent becomes a callable tool
# that the supervisor can invoke like a function.

@tool
def run_triage(transaction_id: str, customer_id: str) -> str:
    """Run fraud triage to quickly classify a transaction's risk level.

    Args:
        transaction_id: The transaction ID to triage
        customer_id: The customer ID who made the transaction
    """
    result = triage_agent(
        f"Triage transaction {transaction_id} for customer {customer_id}."
    )
    return str(result)

@tool
def run_investigation(transaction_id: str, customer_id: str,
                      triage_result: str) -> str:
    """Run a deep investigation on a flagged transaction.

    Args:
        transaction_id: The transaction ID to investigate
        customer_id: The customer who made the transaction
        triage_result: The triage result with risk level and factors
    """
    result = investigation_agent(
        f"Investigate transaction {transaction_id} for customer {customer_id}. "
        f"Triage result: {triage_result}"
    )
    return str(result)

@tool
def run_decision(triage_result: str, investigation_report: str,
                 amount: float) -> str:
    """Render a final fraud verdict based on triage and investigation findings.

    Args:
        triage_result: The triage classification result
        investigation_report: The detailed investigation report
        amount: The transaction amount in dollars
    """
    result = decision_agent(
        f"Render a verdict. Triage: {triage_result}. "
        f"Investigation: {investigation_report}. Amount: ${amount:,.2f}."
    )
    return str(result)

# --- Create the Supervisor Agent ---
supervisor = Agent(
    model=llm,
    tools=[run_triage, run_investigation, run_decision],
    system_prompt=(
        "You are the fraud operations supervisor. "
        "You coordinate fraud investigations by delegating to specialist agents: "
        "1) ALWAYS start with run_triage to classify the risk level "
        "2) If risk is MEDIUM or HIGH, run_investigation to gather evidence "
        "3) If risk is LOW, skip investigation and go straight to run_decision "
        "4) ALWAYS end with run_decision for the final verdict "
        "Report the final verdict clearly to the user."
    ),
)

print("✅ Supervisor agent created with 3 specialist agents as tools!")
print(f"   Tools: run_triage, run_investigation, run_decision")
print(f"\n   Flow: Supervisor -> Triage -> (if needed) Investigation -> Decision")

In [ ]:
# =============================================================================
# DEMO: Run the Full Multi-Agent Fraud Pipeline
# =============================================================================

print("Running multi-agent fraud investigation on TXN-001...")
print("=" * 60)

supervisor_response = supervisor(
    "A customer CUST-001 has a suspicious transaction TXN-001 "
    "— a $4,500 wire transfer to an unknown overseas account at 3:47 AM. "
    "Run the full fraud investigation pipeline."
)

print("=" * 60)
print(f"\nSupervisor's final report:\n{supervisor_response}")

In [ ]:
# =============================================================================
# DEMO: Run on a Known-Legitimate Transaction
# =============================================================================
# For a LOW-risk transaction, the supervisor should skip investigation.

print("Running multi-agent investigation on TXN-002 (Netflix subscription)...")
print("=" * 60)

legit_response = supervisor(
    "Investigate transaction TXN-002 for customer CUST-002 "
    "— a $89.99 payment to Netflix."
)

print("=" * 60)
print(f"\nSupervisor's final report:\n{legit_response}")
print(f"\n💡 Notice: for LOW-risk, the supervisor should skip investigation")
print(f"   and go directly to decision. That's intelligent delegation.")

## Bonus: The Swarm Pattern — Decentralized Collaboration

We just saw the **agents-as-tools** pattern: a supervisor decides the flow.
But what if the agents could decide for themselves?

The **Swarm** pattern gives each agent the ability to **hand off** to any
other agent in the group. There's no central controller — agents collaborate
like a team in a meeting, passing the conversation to whoever is best suited.

| | Supervisor (agents-as-tools) | Swarm |
|---|---|---|
| **Control** | Central supervisor decides flow | Each agent decides who's next |
| **Flow** | Fixed: Triage -> Investigate -> Decide | Emergent: agents hand off freely |
| **Best for** | Predictable pipelines, audit trails | Exploration, complex reasoning |
| **Trade-off** | More control, less flexible | More flexible, harder to predict |

Strands provides `Swarm` from `strands.multiagent`. Each agent automatically
gets `handoff_to_agent()` and `complete_swarm_task()` tools injected — no
extra code needed.

```python
from strands.multiagent import Swarm

swarm = Swarm([agent_a, agent_b, agent_c], entry_point=agent_a)
result = swarm("Investigate TXN-001")
print([node.node_id for node in result.node_history])  # See the flow
```

In [ ]:
# =============================================================================
# BONUS DEMO: Swarm Pattern — Same Agents, Decentralized Flow
# =============================================================================
# Same fraud tools, same specialists — but now THEY decide the flow,
# not a supervisor. Each agent gets handoff_to_agent() and
# complete_swarm_task() injected automatically by the Swarm.

# Create swarm-aware agents (new instances with handoff instructions)
swarm_triage = Agent(
    name="triage",
    model=llm,
    tools=[lookup_transaction, calculate_risk_score],
    system_prompt=(
        "You are a fraud triage specialist. Look up the transaction and "
        "calculate the risk score. "
        "If risk is MEDIUM or HIGH, hand off to 'investigator' with your findings. "
        "If risk is LOW, hand off directly to 'decider' with your findings."
    ),
)

swarm_investigator = Agent(
    name="investigator",
    model=llm,
    tools=[lookup_transaction, check_customer_history, get_similar_transactions],
    system_prompt=(
        "You are a fraud investigator. Gather evidence: check customer history "
        "and search for similar transactions. Write a detailed report. "
        "When done, hand off to 'decider' with your investigation report."
    ),
)

swarm_decider = Agent(
    name="decider",
    model=llm,
    tools=[check_fraud_policy],
    system_prompt=(
        "You are the fraud decision maker. Review the triage and investigation "
        "findings, check the fraud policy, and render a FINAL VERDICT: "
        "APPROVE, FLAG_FOR_REVIEW, or BLOCK. "
        "When you've made your decision, call complete_swarm_task with your verdict."
    ),
)

# Create the swarm — agents decide the flow themselves
fraud_swarm = Swarm(
    [swarm_triage, swarm_investigator, swarm_decider],
    entry_point=swarm_triage,
    max_handoffs=10,
)

# Run it on the same suspicious transaction
print("Swarm investigating TXN-001 (decentralized flow)...")
print("=" * 60)

swarm_result = fraud_swarm(
    "Investigate transaction TXN-001 for potential fraud. "
    "It's a $4,500 wire transfer to an unknown overseas account at 3:47 AM."
)

print("=" * 60)
print(f"\nSwarm status: {swarm_result.status}")
print(f"Agent flow: {' -> '.join(node.node_id for node in swarm_result.node_history)}")
print(f"\nCompare this to the supervisor pattern above — the agents decided")
print(f"the flow themselves based on what they found!")

> **Think About It**: The supervisor agent decided whether to skip investigation
> based on the triage risk level. This is an LLM making a routing decision.
> In production, would you trust the LLM to make this routing decision? Or
> would you prefer hard-coded rules (if risk == "LOW": skip investigation)?
> What are the tradeoffs of each approach?

## Lab 2: Build a Multi-Agent Fraud Pipeline (15 minutes)

### Your Task

Extend the multi-agent pipeline with a 4th specialist agent and test the
full pipeline on a new scenario.

### Steps

1. **Create a compliance agent** — a specialist that checks whether a
   transaction violates any regulatory rules (e.g., transactions over $10,000
   require CTR filing, international wire transfers require OFAC screening).
   Give it a `@tool` called `check_compliance_rules`.
2. **Wrap it as a tool** called `run_compliance_check` using the agents-as-tools pattern.
3. **Update the supervisor** — add the compliance tool and update the system
   prompt so the supervisor always runs compliance check after investigation
   but before decision.
4. **Test the full pipeline** on transaction TXN-001 ($4,500 wire transfer
   to overseas). Does the compliance agent flag the OFAC requirement?

### Expected Output

- A working compliance agent with regulatory rules
- Updated supervisor with 4 specialist agents
- A final verdict that includes compliance findings

### Homework Extension

After class: add error handling to the pipeline. What happens if the
triage agent fails or returns an invalid risk level? Implement a fallback
that routes directly to human review when any agent in the pipeline errors.

In [ ]:
# =============================================================================
# LAB 2: MULTI-AGENT FRAUD PIPELINE WITH COMPLIANCE
# =============================================================================

# Step 1: Create compliance rules tool
@tool
def check_compliance_rules(amount: float, is_international: bool,
                           destination_country: str) -> str:
    """Check regulatory compliance rules for a transaction.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction crosses borders
        destination_country: The destination country for the transaction
    """
    result = None  # YOUR CODE

    return json.dumps(result, indent=2)


# Step 2: Create the compliance agent
compliance_agent = None  # YOUR CODE


# Step 3: Wrap as tool
@tool
def run_compliance_check(amount: float, is_international: bool,
                         destination_country: str) -> str:
    """Run regulatory compliance check on a transaction.

    Args:
        amount: Transaction amount in dollars
        is_international: Whether the transaction is international
        destination_country: Destination country
    """
    result = None  # YOUR CODE

    return str(result)


# Step 4: Create updated supervisor with compliance
updated_supervisor = None  # YOUR CODE


# Step 5: Test
test_result = None  # YOUR CODE

print(test_result)

# Section 3: AgentCore Memory — Persistent Agent Knowledge

## From Ephemeral to Persistent Memory

In Week 15, we used LangGraph's `MemorySaver`:
- Simple: `MemorySaver()` + `thread_id`
- **Ephemeral**: Memory dies when the notebook restarts
- **In-process**: Can't share memory between agents or services
- **No long-term learning**: Each session starts from scratch

**Amazon Bedrock AgentCore Memory** solves all of these:
- **Persistent**: Stored in AWS — survives restarts, deployments, scaling
- **Shared**: Any agent (or service) can read/write to the same memory
- **Short-term + Long-term**: Session turns (conversation) + extracted knowledge
- **Searchable**: Semantic search over stored memories
- **Managed**: No database to maintain, automatic scaling

## Memory Types

| Type | What it stores | Use case |
|------|---------------|----------|
| **Short-term** | Turn-by-turn conversation | "What did the user just say?" |
| **Long-term (Semantic)** | Extracted entities and facts | "This customer prefers email notifications" |
| **Long-term (Summary)** | Session summaries | "Last time, we investigated 3 wire transfers" |
| **Long-term (Episodic)** | Structured episodes | "TXN-001 was blocked due to OFAC violation" |

In [ ]:
# =============================================================================
# DEMO: AgentCore Memory — Agent That Actually Remembers Across Turns
# =============================================================================
# In Week 15, MemorySaver gave agents memory — but it died when the notebook
# restarted. AgentCore Memory is PERSISTENT: it survives restarts, deployments,
# and can be shared across agents and services.
#
# We'll create an agent with AgentCoreMemorySessionManager — it automatically
# stores every turn in AgentCore Memory and retrieves past context.

# Step 1: Create or find a Memory resource
control_client = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)
MEMORY_ID = None
MEMORY_NAME = "week16-fraud-investigation"

try:
    memories = control_client.list_memories().get('memories', [])
    existing = next((m for m in memories if m['name'] == MEMORY_NAME), None)

    if existing:
        MEMORY_ID = existing['id']
        print(f"Found existing memory: {MEMORY_NAME} (id: {MEMORY_ID})")
    else:
        response = control_client.create_memory(
            name=MEMORY_NAME,
            description="Fraud investigation memory for Week 16 agents",
            eventExpiryDuration=86400,  # 24 hours
        )
        MEMORY_ID = response['memory']['id']
        print(f"Created new memory: {MEMORY_NAME} (id: {MEMORY_ID})")
except Exception as e:
    print(f"AgentCore Memory not available: {e}")
    print("This is OK — see the code pattern below for reference.")

# Step 2: Create an agent WITH persistent memory
if MEMORY_ID:
    # Configure the memory session — same memory_id + session_id = shared context
    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id="investigation-session-demo",
        actor_id="fraud-analyst",
    )

    # Use context manager to ensure buffered messages are flushed
    with AgentCoreMemorySessionManager(
        agentcore_memory_config=config,
        region_name=AWS_REGION,
    ) as memory_session:

        # Create agent with persistent memory — turns auto-stored/retrieved
        memory_agent = Agent(
            model=llm,
            tools=fraud_tools,
            system_prompt=(
                "You are a senior fraud analyst with persistent memory. "
                "When investigating, use your tools thoroughly. "
                "Remember findings from previous turns in this session."
            ),
            session_manager=memory_session,
        )

        # Turn 1: Investigate TXN-001 — findings automatically stored in memory
        print("TURN 1: Investigating TXN-001...")
        print("=" * 60)
        result1 = memory_agent("Investigate transaction TXN-001. Is it fraud?")
        print(str(result1)[:500])

        # Turn 2: Follow-up — agent remembers Turn 1 from AgentCore Memory
        print("\n\nTURN 2: Follow-up (agent should remember without re-investigating)...")
        print("=" * 60)
        result2 = memory_agent(
            "What was the risk level of the transaction we just investigated? "
            "What fraud policy applies to it?"
        )
        print(str(result2)[:500])

    print(f"\n✅ Both turns stored in AgentCore Memory!")
    print(f"   Memory ID: {MEMORY_ID}")
    print(f"   Session: investigation-session-demo")
    print(f"\n   Unlike Week 15's MemorySaver, this persists across notebook restarts.")
    print(f"   Any agent with the same memory_id + session_id can read these findings.")

else:
    # Fallback: show the code pattern
    print("\nCode pattern (for reference):")
    print()
    print("  config = AgentCoreMemoryConfig(")
    print("      memory_id='your-memory-id',")
    print("      session_id='investigation-1',")
    print("      actor_id='fraud-analyst',")
    print("  )")
    print()
    print("  with AgentCoreMemorySessionManager(config, region_name=AWS_REGION) as sm:")
    print("      agent = Agent(model=llm, tools=fraud_tools, session_manager=sm)")
    print("      agent('Investigate TXN-001')  # auto-stored in memory")
    print("      agent('What was the verdict?')  # remembers from memory")
    print()
    print("Key difference from Week 15's MemorySaver:")
    print("   MemorySaver: in-process, dies when notebook restarts")
    print("   AgentCore Memory: persistent, shared, searchable, managed by AWS")

In [ ]:
# =============================================================================
# DEMO: The Full AgentCore Production Stack
# =============================================================================
# Let's map what we've learned to the production architecture.

print("Amazon Bedrock AgentCore — Production Architecture")
print("=" * 60)
print()
print("  +-----------------------------------------------------+")
print("  |                AgentCore Runtime                     |")
print("  |  (Hosts your agents in managed containers)           |")
print("  |                                                      |")
print("  |  +-----------+  +-----------+  +-----------+        |")
print("  |  | Triage    |  | Investigate|  | Decision  |        |")
print("  |  | Agent     |  | Agent     |  | Agent     |        |")
print("  |  | (Strands) |  | (Strands) |  | (Strands) |        |")
print("  |  +-----+-----+  +-----+-----+  +-----+-----+        |")
print("  |        |              |              |               |")
print("  |  +-----+--------------+--------------+-----+        |")
print("  |  |           Supervisor Agent              |        |")
print("  |  +--------------------+--------------------+        |")
print("  +----------------------|------------------------+")
print("                         |")
print("    +-----------+--------+--------+-----------+")
print("    |           |                 |           |")
print("  +-+------+  +-+------+  +------+-+  +------+-+")
print("  | Memory |  |Gateway |  |Identity|  |Observ- |")
print("  |(persis-|  |(MCP    |  |(creds  |  |ability |")
print("  | tent)  |  | tools) |  | mgmt)  |  |(traces)|")
print("  +--------+  +--------+  +--------+  +--------+")
print()
print("What we covered today:")
print("  Strands Agents SDK — build agents (Section 1)")
print("  Multi-agent orchestration — agents-as-tools (Section 2)")
print("  AgentCore Memory — persistent knowledge (Section 3)")
print()
print("What's in the optional notebook:")
print("  AgentCore Runtime — deploy agents to managed containers")
print("  AgentCore Gateway — convert APIs to MCP tools")
print("  AgentCore Identity — secure credential management")
print("  AgentCore Observability — tracing and monitoring")

> **Think About It**: We built a multi-agent fraud pipeline that runs in a
> notebook. In production, this pipeline would need to handle thousands of
> transactions per day, with audit trails, failover, and compliance logging.
> Which AgentCore services (Runtime, Memory, Gateway, Identity, Observability)
> would be most critical to set up first? Why?

## Connecting the Dots: Agents + Fine-Tuned Models

Remember Week 14's fine-tuned DistilBERT fraud classifier? In a production
multi-agent system, you could use it as a **fast pre-filter**:

```python
@tool
def classify_with_distilbert(description: str) -> str:
    """Run the fine-tuned DistilBERT fraud classifier on a transaction."""
    from transformers import pipeline
    classifier = pipeline("text-classification", model="./fraud-classifier")
    result = classifier(description)[0]
    return json.dumps({"prediction": result["label"], "confidence": result["score"]})
```

**The production architecture**:
1. DistilBERT classifies first (FREE, fast, runs locally)
2. Only MEDIUM/HIGH confidence fraud cases go to the full agent pipeline
3. This saves API costs — the LLM agent only handles complex cases

This is a homework extension — try it after class!

## Looking Ahead

- **Weeks 17-18**: RAG (Retrieval-Augmented Generation) — give agents access to
  document knowledge bases. Your AgentCore Gateway can connect to Bedrock
  Knowledge Bases as MCP tools.
- **Weeks 19-20**: MLOps — how to version, monitor, and maintain your agent
  pipelines in production.

# Summary: What We Learned Today

## Key Takeaways

### Strands Agents SDK
- AWS's production agent framework — simpler API than LangChain
- `Agent(model=..., tools=[...], system_prompt="...")` — one clean constructor
- `@tool` decorator works just like LangChain's — concepts are portable
- `from strands.models import BedrockModel` for Bedrock integration

### Multi-Agent Orchestration
- **Agents-as-tools pattern**: wrap specialist agents in `@tool`, give to supervisor
- `callback_handler=None` suppresses sub-agent output for clean supervisor responses
- Supervisor makes routing decisions (skip investigation for LOW risk)
- Each specialist has its own tools and expertise

### AgentCore Memory
- Persistent, cross-session memory (unlike Week 15's ephemeral MemorySaver)
- `MemorySessionManager` -> `create_memory_session` -> `add_turns` / `get_last_k_turns`
- Short-term (conversation) + long-term (semantic, summary, episodic)
- Managed by AWS — no database to maintain

### Production Architecture
- AgentCore = Runtime + Memory + Gateway + Identity + Observability
- Framework-agnostic: works with Strands, LangGraph, CrewAI
- GA since October 2025, available in 9 AWS regions

# Homework and Optional Labs

## Homework (Complete before next session)

### Homework 1: DistilBERT as Agent Tool
Wrap your Week 14 fine-tuned DistilBERT model as a Strands `@tool`.
Build a pipeline where DistilBERT pre-screens transactions and only
sends ambiguous cases to the full multi-agent pipeline.

### Homework 2: Memory-Enriched Agent
If you have access to AgentCore Memory, build an agent that:
- Stores investigation results in AgentCore Memory
- Before investigating a new transaction, searches memory for similar
  past investigations
- Uses past findings to inform current investigation

## Optional Lab: AgentCore Runtime Deployment
See `week_16_optional_agentcore_runtime.ipynb` for a walkthrough of:
- Packaging your Strands agent for AgentCore Runtime
- Deploying with the starter toolkit CLI
- Invoking your deployed agent via boto3
- Setting up AgentCore Gateway for MCP tool integration

# Great Work Today!

You've completed Week 16 of the AI for Data Scientists Academy.

**The journey so far:**
- Weeks 11-12: Prompting LLMs (OpenAI, HuggingFace) — making models respond
- Week 13: Amazon Bedrock — enterprise-grade model access
- Week 14: Fine-tuning — training your own models
- Week 15: Single agents — making models ACT (tools, memory, ReAct)
- **Week 16: Multi-agent systems and production deployment** — making agents
  COLLABORATE at scale with Strands + AgentCore

**Next up**: Weeks 17-18 — RAG (Retrieval-Augmented Generation)